1. Basic Chat Models
• 	Simple LLM call: Using  to send a prompt and get a response.
• 	Prompt Templates: Reusable prompt structures for consistent outputs.
• 	Chains: Combine multiple steps (e.g., prompt → LLM → output parsing).

In [ ]:
!pip uninstall -y langchain langchain-core langchain-openai langchain-community networkx faiss-cpu
!pip install langchain==0.1.17
!pip install langchain-openai==0.0.6
!pip install langchain-core==0.1.53 
!pip install langchain-community==0.0.33
!pip install networkx
!pip install pypdf
!pip install faiss-cpu
!pip install fastapi uvicorn

In [ ]:
from langchain_openai import ChatOpenAI

chat = ChatOpenAI(
    model="gpt-4o-mini",
    temperature="0.7"
)
response= chat.invoke("What is LangChain?");
print(response.content);

In [ ]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.docstore.document import Document

# 1. Prepare documents
docs = [
    Document(page_content="Python is a programming language."),
    Document(page_content="OpenAI provides GPT models for text generation."),
]

# 2. Split text into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)

# 3. Create embeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 4. Store in FAISS
vectorstore = FAISS.from_documents(chunks, embeddings)

# 5. Setup retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 6. Setup generator (LLM)
llm = ChatOpenAI(model="gpt-4.1-mini")

# 7. Build RAG pipeline
qa_chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever)

# 8. Ask a question
query = "What is Python used for?"
answer = qa_chain.invoke(query)

print(answer)

2. Memory-Enabled Conversations
• 	ConversationBufferMemory: Stores past messages for context.
• 	ConversationSummaryMemory: Summarizes past interactions to keep context lightweight.
• 	ConversationKGMemory: Builds a knowledge graph from conversations.

1. ConversationBufferMemory Stores the entire conversation history verbatim.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# Define a prompt with history
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),  # chat history placeholder
    ("human", "{input}"),                          # matches your 'input' key
])

# Add buffer memory
memory = ConversationBufferMemory(return_messages=True)

# Build conversation chain
conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True
)

# Example interactions
print(conversation.invoke("Hello, my name is Pankaj."))
print(conversation.invoke("What is my name?"))

2. ConversationSummaryMemory Summarizes past interactions instead of storing them verbatim (useful for long chats).

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.chains import ConversationChain
from langchain.memory import ConversationSummaryMemory

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# Add summary memory
memory = ConversationSummaryMemory(llm=llm)

conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True
)

print(conversation.run("I am working on a Python project."))
print(conversation.run("Remind me what I said earlier."))

3. ConversationKGMemory: Builds a knowledge graph from conversations.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.chains import ConversationChain
from langchain.memory import ConversationKGMemory

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# Add knowledge graph memory
memory = ConversationKGMemory(llm=llm)

conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True
)

print(conversation.run("Alice is Bob's friend."))
print(conversation.run("Who is Bob's friend?"))

RAG Example with FAISS

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.document_loaders import PyPDFLoader

# 1. Load documents (PDF example)
loader = PyPDFLoader("HR_Policy_Manual_2023.pdf")
docs = loader.load()

# 2. Split into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(docs)

# 3. Create embeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 4. Store in FAISS vector DB
vectorstore = FAISS.from_documents(documents, embeddings)

# 5. Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 6. Build RetrievalQA chain
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",   # simple chain type
    retriever=vectorstore.as_retriever(),
    return_source_documents=True
)

# 7. Ask questions
query = "What does the document say about LangChain?"
result = qa.invoke(query)

print("Answer:", result["result"])
print("Sources:", result["source_documents"])

RAG Agent (multi-tool reasoning)
If you want the model to decide when to use retrieval vs. direct reasoning, you can wrap the retriever as a tool and use an Agent:


In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_react_agent, AgentExecutor
from langchain.prompts import PromptTemplate
from langchain.tools import Tool

# LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# Tool
def summarize_text(text: str) -> str:
    return "• Point 1\n• Point 2\n• Point 3"

summarize_tool = Tool(
    name="Summarizer",
    func=summarize_text,
    description="Summarizes text into 3 bullet points."
)

tools = [summarize_tool]

# ReAct prompt
prompt = PromptTemplate.from_template("""
You are a helpful assistant.

You have access to the following tools:
{tools}

Use the following format:

Question: {input}
Thought: think about what to do
Action: one of [{tool_names}]
Action Input: input to the tool
Observation: result of the tool
Thought: I now know the final answer
Final Answer: your answer here

{agent_scratchpad}
""")

# Agent
agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)

# Executor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# Run
result = agent_executor.invoke({"input": "Summarize the document in 3 bullet points."})
print(result["output"])

4. Agents
- Tool-Using Agents: LLMs that can call external tools (search, calculator, APIs).
- Multi-Tool Agents: Combine multiple tools for complex workflows.
- Custom Tools: Define your own functions and integrate them into agents.

1. Tool-Using Agent

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, Tool
from langchain.agents import AgentType

# Define a simple calculator tool
def calculator_tool(query: str) -> str:
    try:
        return str(eval(query))
    except Exception as e:
        return f"Error: {e}"

tools = [
    Tool(
        name="Calculator",
        func=calculator_tool,
        description="Useful for math calculations."
    )
]

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

print(agent.run("What is 25 * 4 + 10?"))

 Multi-Tool Agent (Search + Calculator)

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, Tool, AgentType

# Define calculator
def calculator_tool(query: str) -> str:
    try:
        return str(eval(query))
    except Exception as e:
        return f"Error: {e}"

# Define a dummy search tool
def search_tool(query: str) -> str:
    return f"Search results for: {query}"

tools = [
    Tool(name="Calculator", func=calculator_tool, description="Math calculations"),
    Tool(name="Search", func=search_tool, description="Search the web")
]

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

print(agent.run("Search for LangChain and then calculate 12*8"))

Custom Tool Agent You can define any Python function as a tool and integrate it:

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, Tool, AgentType

# Custom tool: reverse a string
def reverse_text(query: str) -> str:
    return query[::-1]

tools = [
    Tool(
        name="ReverseText",
        func=reverse_text,
        description="Reverses the input string."
    )
]

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

print(agent.run("Reverse the word LangChain"))

- Integrations
- Databases: Connect to SQL databases for natural language querying.
- APIs: Wrap external APIs (weather, finance, etc.) into LangChain tools.
- Cloud Services: Integrate with AWS, Azure, or Google Cloud for deployment.


Databases: Connect to SQL databases for natural language querying.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.sql_database import SQLDatabase
from langchain.chains import SQLDatabaseChain

# Connect to a local SQLite database
db = SQLDatabase.from_uri("sqlite:///my_database.db")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Build SQL chain
db_chain = SQLDatabaseChain.from_llm(llm, db, verbose=True)

# Ask natural language questions
print(db_chain.run("How many users signed up last month?"))

APIs: Wrap external APIs (weather, finance, etc.) into LangChain tools.

In [ ]:
import requests
from langchain_openai import ChatOpenAI
from langchain.agents import create_openai_functions_agent, AgentExecutor, tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Define a tool
@tool
def weather_tool(city: str) -> str:
    """Get current weather for a city."""
    response = requests.get(f"https://wttr.in/{city}?format=3")
    return response.text

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Define prompt with both input and agent_scratchpad
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that can use tools."),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# Create agent
tools = [weather_tool]
agent = create_openai_functions_agent(llm=llm, tools=tools, prompt=prompt)

# Wrap in executor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# Run the agent
result = agent_executor.invoke({"input": "What's the weather in Pune?"})
print(result)

Cloud Services (Deployment)
LangChain apps can be deployed on AWS, Azure, or GCP. Typical patterns:
- AWS Lambda + API Gateway → Wrap your LangChain pipeline as a serverless function.
- Azure Functions → Similar serverless deployment with integration to Azure OpenAI.
- GCP Cloud Run → Containerize your LangChain app with Docker and deploy.


In [ ]:
from fastapi import FastAPI
from langchain_openai import ChatOpenAI

app = FastAPI()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

@app.get("/ask")
def ask(query: str):
    response = llm.invoke(query)
    return {"answer": response.content}